In [ ]:
# Load the data and build the search index:

from rag_helper import RAGBase
from ingest import load_faq_data, build_index

documents = load_faq_data()
index = build_index(documents)


In [ ]:
# Set up the OpenAI client:

import os
from dotenv import load_dotenv
load_dotenv()


from openai import OpenAI
openai_client = OpenAI(api_key=os.getenv("OPEN_API_KEY"))


In [ ]:
# Create the assistant:

instructions = """
You're a course teaching assistant.
Answer the QUESTION based on the CONTEXT from the FAQ database.
Use only the facts from the CONTEXT when answering the QUESTION.
""".strip()

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
    instructions=instructions,
)

In [ ]:
# This works fine. The search finds relevant FAQ entries about Ollama, and the LLM gives a good answer.

assistant.rag("How do I run Ollama locally?")

'To run Ollama locally, follow these steps:\n\n1. **Install Ollama**:\n   - For **macOS**, download the `.pkg` installer from [https://ollama.com/download](https://ollama.com/download) and install it.\n   - For **Windows**, download the `.msi` installer from the same link and install it.\n   - For **Linux**, open a terminal and run:\n     ```bash\n     curl -fsSL https://ollama.com/install.sh | sh\n     ```\n\n2. **Run the Ollama model locally**:\n   Open a terminal and execute:\n   ```bash\n   ollama run llama3\n   ```\n   This will download the LLaMA 3 model (~4GB), start it locally, and open a chat-like interface for interactions.\n\n3. **Test the Ollama local server**:\n   Run:\n   ```bash\n   curl http://localhost:11434\n   ```\n   You should get a JSON response listing available models.\n\n4. **Use the Python client**:\n   Install it with:\n   ```bash\n   pip install ollama\n   ```\n   Then you can interact with the model via Python, for example:\n   ```python\n   import ollama\n

In [10]:
# The word "Olama" doesn't match "Ollama" in our index. We use lexical search, so it looks for the exact word and finds nothing. 
# The LLM gets these bad results and either says "I don't know" or answers with irrelevant information.

assistant.rag("How do I run Olama locally?")

'You can run Olama locally if you are comfortable setting up the necessary tools such as Python, `uv`, Jupyter, Docker, and any other required components. While Codespaces provides an easy and consistent environment for everyone, running locally is possible as long as you properly document your setup and keep your environment reproducible.'

In [ ]:
# This is the limitation of a fixed pipeline. 
# The search runs once with the exact query the user typed, and there's no second chance. The pipeline doesn't know the search failed, 
# so it can't try again with a corrected query.

# We need something smarter. We need an agent.


```mermaid
flowchart TD
    U([User: How do I run Olama?])
    S[search - Olama - no useful results]
    A([LLM: I don't have information about Olama.])

    U --> S --> A
```

In [ ]:
# The agent alternative:

# An agent puts the LLM in charge.
# Instead of running search ourselves, we give the LLM a 'search' tool. It decides when to call it and what to search for.


# The same typo uestion now goes like this:

```mermaid
flowchart TD
    U([User: How do I run Olama?])
    L1[LLM: I'll search for 'Olama']
    S1[search - Olama - no useful results]
    L2[LLM: Hmm, no results. Maybe a typo for 'Ollama'?]
    S2[search - Ollama - found results!]
    A([LLM: Here's how to run Ollama locally...])

    U --> L1 --> S1 --> L2 --> S2 --> A
```

In [ ]:
# The LLM searched, saw the results were bad, and decided to try again with a different query.
# It made that decision on its own. We didn't write any code to handle typos.

# The difference is about who makes the decision:
#   - With RAG, the developer decides. We fix the steps up front, so search always runs once with the exact user query.
#   - With an agent, the LLM decides. It chooses which actions to take and when to stop

# The mechanism that makes this possible is function calling, and that's what the rest of this lesson is about.

In [14]:
## Asking without any tools:

# First, let's see what the LLM does without any tools. We ask it a course-specific questions and look at the answer

messages = [
    {"role": "user", "content": "I just discovered the course. Can I join it?"}
]

response = openai_client.responses.create(
    model="gpt-4.1-mini",
    input=messages
)

response.output_text


# The model answers from its general knowledge. It doesn't know about our FAQ, so the answer is vague and not helpful.
# This is exactly why we need RAG, and why we want to hand the model a tool

'I’d be happy to help! Could you please specify which course you’re referring to? That way, I can provide you with the most accurate information about joining.'

In [15]:
## Defining the tool

# First we define a top-level 'search' function that queries the 'index' directly.
# The model will reference it by this name. We keep the Python function and the tool name aligned, so the dispatch is easier later

def search(query):
        boost_dict={"question": 2.0, "section": 0.5} # Boost the question field twice as much as the section field
        filter_dict={"course": "llm-zoomcamp"} # Filter results to only include documents from the specified course

        return index.search(
            query, 
            boost_dict=boost_dict,
            filter_dict=filter_dict,
            num_results=5
        )

In [ ]:
# Next we tell the model about this function.
# The model doesn't see our Python code, it sees only a schema describing what the function does and what arguments it takes.

search_tool = {
    "type": "function",
    "name": "search",
    "description": "Search the FAQ database for entries matching the given query.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "Search query text to look up in the course FAQ."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    }
}

# Description - Is the most important field, because the model reads it to decide when to call the function
# parameters - is a JSON schema for the arguments
# Required - We make query as required

In [ ]:
## Sending the question with the tool

# Now we send the same question as before, but this time we include the tool in the request:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool],
)

response.output

# Look at the response. Instead of a message with the answer, the response contains a 'function call' entry.
# The model decided it needs to search the FAQ before answering. Rather than reply, it asked us to run the search function first

# Look at the arguments too. The model didn't pass our question verbatim (Exactly as it was spoken or written)
# It judged the raw question wasn't the best query to search with. So it rewrote our enrollment question into search keywords like "enroll late join course".

[ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join enrollment late registration"}', call_id='call_8hV8UiVdVQAas021UqpkMmDk', name='search', type='function_call', id='fc_0680c11274e98aa7006a425a8172588196bea6010575cea099', namespace=None, status='completed')]

In [ ]:
## Executing the function and sending the result back

# The function call contains JSON arguments. We parse them, call our 'search' function, and serialize the result.

import json

call = response.output[0]
args = json.loads(call.arguments)

results = search(**args)
result_json = json.dumps(results, indent=2)

# Now we send this result back to the model
# First, we add the model's output to the conversation history - the model needs to see its own function call. Then we add the tool result

messages.extend(response.output)

messages.append({
    "type": "function_call_output",
    "call_id": call.call_id,
    "output": result_json,
})

# The call_id links the tool result to the specific function call the model requested. If the model makes multiple function calls in one turn, each one gets its own call_id.

messages


[{'role': 'user', 'content': 'I just discovered the course. Can I join it?'},
 ResponseFunctionToolCall(arguments='{"query":"Can I join the course late discovered course join enrollment late registration"}', call_id='call_8hV8UiVdVQAas021UqpkMmDk', name='search', type='function_call', id='fc_0680c11274e98aa7006a425a8172588196bea6010575cea099', namespace=None, status='completed'),
 {'type': 'function_call_output',
  'call_id': 'call_8hV8UiVdVQAas021UqpkMmDk',
  'output': '[\n  {\n    "id": "74eb249bbf",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "I just discovered the course. Can I still join?",\n    "answer": "Yes, but if you want to receive a certificate, you need to submit your project while we\\u2019re still accepting submissions."\n  },\n  {\n    "id": "977bf7786c",\n    "course": "llm-zoomcamp",\n    "section": "General Course-Related Questions",\n    "question": "Course: I have registered for the LLM Zoomcamp. When can I ex

In [ ]:
## Asking the model again

# We call the API a second time with the expanded history:

response = openai_client.responses.create(
    model="gpt-5.4-mini",
    input=messages,
    tools=[search_tool]
)

response.output_text


# This time the model has the original question, its own decision to call search, and the FAQ results. It can now produce a proper course-specific answer.

# We have to send the whole history because LLMs are stateless between API calls. The memory is the list you send as input. 
# If you send only the tool result, the model has no idea what's going on. So on this second call we replay everything we have so far. 
# That means the question, the decision to call search, and the result we got back.

# That's the full function-calling loop for a single turn. With plain RAG we made one call, and here we make two. Turning RAG agentic means more round-trips.

# People call this pattern "agentic RAG", "tool use", or "function calling". The idea behind all of them is the same. The LLM decides which tools to call.


'Yes — you can still join and start learning.\n\nIf you want a certificate, make sure to submit your project while submissions are still open.'

In [26]:
# Token usage and cost

usage = response.usage
usage.input_tokens, usage.output_tokens

# For each model the provider publishes a price per million input tokens and per million output tokens. Plug those numbers in to convert tokens to dollars.


def calculate_gpt54mini_price(input_tokens, output_tokens):
    INPUT_PRICE_PER_MILLION = 0.15
    OUTPUT_PRICE_PER_MILLION = 0.60

    input_cost = (input_tokens / 1_000_000) * INPUT_PRICE_PER_MILLION
    output_cost = (output_tokens / 1_000_000) * OUTPUT_PRICE_PER_MILLION
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
    }

result = calculate_gpt54mini_price(652, 33)
print("Total cost: $", round(result["total_cost"], 8))


Total cost: $ 0.0001176


In [27]:
# gpt-4.1-mini

input_price = 0.40 / 1_000_000  # $0.40 per 1M tokens for gpt-4.1-mini
output_price = 1.60 / 1_000_000  # $1.60 per 1M tokens for gpt-4.1-mini

cost = (
    response.usage.input_tokens * input_price +
    response.usage.output_tokens * output_price
)

cost

0.00036040000000000003